In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

df = pd.read_csv('Alternative.csv')

# 2. 피처 엔지니어링 (Marketing Synergy 적용)
X = df[['TV', 'radio', 'newspaper']]
y = df['sales']

# 상호작용항(Interaction) 생성: TV * Radio 시너지 효과 추적
# degree=2, interaction_only=True는 단독 매체 효과와 '두 매체의 조합 효과'만 뽑아냅니다.
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_poly = poly.fit_transform(X)
poly_columns = poly.get_feature_names_out(X.columns)

# 3. 데이터 분할 및 스케일링
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42)

# 단위가 다른 매체비를 공정하게 비교하기 위한 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Ridge 회귀 모델 학습 (규제 적용으로 모델 안정성 확보)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)

# 5. 결과 해석 테이블 생성
coef_df = pd.DataFrame({
    '변수명(매체 및 조합)': poly_columns,
    '매출기여도(Coefficient)': ridge_model.coef_
}).sort_values(by='매출기여도(Coefficient)', ascending=False)

print("\n🚀 [분석 완료] 매체별 단독 효과 및 시너지 분석 결과")
print("-" * 60)
print(coef_df)
print("-" * 60)
print(f"📊 모델의 매출 설명력(R-squared): {r2_score(y_test, ridge_model.predict(X_test_scaled)):.4f}")


🚀 [분석 완료] 매체별 단독 효과 및 시너지 분석 결과
------------------------------------------------------------
      변수명(매체 및 조합)  매출기여도(Coefficient)
3         TV radio            3.510619
0               TV            1.761130
1            radio            0.527695
2        newspaper            0.296262
5  radio newspaper           -0.119551
4     TV newspaper           -0.250750
------------------------------------------------------------
📊 모델의 매출 설명력(R-squared): 0.9744
